# Databricks/PySpark dla Data Analyst/Engineer — Moduł 12: MLlib -- podstawy

Wracamy do w pełni wykonywalnego kodu. **MLlib** to biblioteka uczenia maszynowego
Sparka -- te same modele, które poznaliśmy w kursie Statystyki (regresja liniowa i
logistyczna, Moduły 12-13), tylko teraz trenowane w sposób ROZPROSZONY, na danych
większych niż zmieściłyby się w pamięci jednego komputera.

## Spis treści
1. [MLlib: przegląd DataFrame-based API](#sec1)
2. [Przygotowanie cech: VectorAssembler](#sec2)
3. [Zmienne kategoryczne: StringIndexer i OneHotEncoder](#sec3)
4. [Pipeline: łączenie kroków w jeden obiekt](#sec4)
5. [Regresja liniowa: przewidywanie wartości zamówienia](#sec5)
6. [Regresja logistyczna: przewidywanie anulowania zamówienia](#sec6)
7. [Podsumowanie i ćwiczenia](#sec7)


Uruchamiamy (albo pobieramy już istniejącą) `SparkSession` — punkt wejścia do wszystkiego w Sparku. W Databricks `spark` jest już gotowe automatycznie w każdym notebooku; ten sam kod działa też lokalnie.

In [ ]:
from pyspark.sql import SparkSession

# Dodatkowe opcje JVM (--add-opens) sa potrzebne, gdy uruchamiasz Sparka lokalnie na
# Javie 17+ i chcesz uzywac pandas UDF / Apache Arrow (Modul 8) -- nowsze JDK domyslnie
# blokuja refleksyjny dostep do pamieci poza stosem, ktorego potrzebuje Arrow. W
# Databricks nie trzeba tego ustawiac recznie -- platforma robi to za Ciebie.
_opcje_jvm = (
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED"
)

spark = (
    SparkSession.builder.appName("kurs_spark")
    .config("spark.driver.extraJavaOptions", _opcje_jvm)
    .config("spark.executor.extraJavaOptions", _opcje_jvm)
    .getOrCreate()
)
spark


Wczytujemy nasz wspólny zestaw danych — dane sklepu internetowego (`klienci`, `produkty`, `pracownicy`, `zamowienia`, `pozycje_zamowien`), dokładnie ten sam schemat co w kursie SQL, ale tym razem jako DataFrame'y Sparka.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)

# --- klienci -----------------------------------------------------------
n_klienci = 300
miasta = ["Warszawa", "Krakow", "Gdansk", "Wroclaw", "Poznan"]
segmenty = ["Nowy", "Standardowy", "Premium"]
prawdop_segmentu = [0.35, 0.45, 0.20]

klienci_pd = pd.DataFrame({
    "klient_id": np.arange(1, n_klienci + 1),
    "miasto": rng.choice(miasta, size=n_klienci, p=[0.30, 0.25, 0.15, 0.15, 0.15]),
    "segment": rng.choice(segmenty, size=n_klienci, p=prawdop_segmentu),
    "dni_od_rejestracji": rng.integers(1, 900, size=n_klienci),
})

# --- produkty ------------------------------------------------------------
n_produkty = 40
kategorie = ["Elektronika", "Odziez", "Dom", "Sport", "Ksiazki"]
srednie_ceny = {"Elektronika": 1200, "Odziez": 150, "Dom": 250, "Sport": 300, "Ksiazki": 50}

kategorie_produktow = rng.choice(kategorie, size=n_produkty)
produkty_pd = pd.DataFrame({
    "produkt_id": np.arange(1, n_produkty + 1),
    "kategoria": kategorie_produktow,
    "cena": [
        round(max(5.0, rng.normal(srednie_ceny[k], srednie_ceny[k] * 0.3)), 2)
        for k in kategorie_produktow
    ],
})

# --- pracownicy ----------------------------------------------------------
n_pracownicy = 12
regiony = ["Polnoc", "Poludnie", "Wschod", "Zachod"]
pracownicy_pd = pd.DataFrame({
    "pracownik_id": np.arange(1, n_pracownicy + 1),
    "region": rng.choice(regiony, size=n_pracownicy),
    "dzial": rng.choice(["Sprzedaz", "Obsluga"], size=n_pracownicy, p=[0.6, 0.4]),
})

# --- zamowienia ------------------------------------------------------------
n_zamowienia = 3000
klient_id_zamowien = rng.choice(klienci_pd["klient_id"], size=n_zamowienia)
pracownik_id_zamowien = rng.choice(pracownicy_pd["pracownik_id"], size=n_zamowienia)

segment_mapa = klienci_pd.set_index("klient_id")["segment"]
segment_zamowien = klient_id_zamowien.astype(object)
segment_zamowien = pd.Series(klient_id_zamowien).map(segment_mapa).values

# prawdopodobienstwo anulowania zalezy od segmentu klienta (celowo wbudowany efekt)
prawdop_anulowania = np.where(
    segment_zamowien == "Nowy", 0.22,
    np.where(segment_zamowien == "Standardowy", 0.10, 0.04),
)
anulowane = rng.binomial(1, prawdop_anulowania)
status_zamowien = np.where(
    anulowane == 1, "Anulowane",
    rng.choice(["Zlozone", "Wyslane", "Dostarczone"], size=n_zamowienia, p=[0.2, 0.3, 0.5]),
)

data_bazowa = pd.Timestamp("2024-01-01")
dni_offset = rng.integers(0, 730, size=n_zamowienia)

zamowienia_pd = pd.DataFrame({
    "zamowienie_id": np.arange(1, n_zamowienia + 1),
    "klient_id": klient_id_zamowien,
    "pracownik_id": pracownik_id_zamowien,
    "data_zamowienia": [data_bazowa + pd.Timedelta(days=int(d)) for d in dni_offset],
    "status": status_zamowien,
})

# --- pozycje_zamowien ------------------------------------------------------
wiersze_pozycji = []
pozycja_id = 1
cena_produktu = produkty_pd.set_index("produkt_id")["cena"]

for zamowienie_id in zamowienia_pd["zamowienie_id"]:
    liczba_pozycji = rng.integers(1, 5)
    wybrane_produkty = rng.choice(produkty_pd["produkt_id"], size=liczba_pozycji, replace=False)
    for produkt_id in wybrane_produkty:
        ilosc = int(rng.integers(1, 4))
        rabat = rng.choice([0.0, 0.05, 0.10, 0.15], p=[0.55, 0.20, 0.15, 0.10])
        wiersze_pozycji.append({
            "pozycja_id": pozycja_id,
            "zamowienie_id": int(zamowienie_id),
            "produkt_id": int(produkt_id),
            "ilosc": ilosc,
            "cena_jednostkowa": float(cena_produktu[produkt_id]),
            "rabat": float(rabat),
        })
        pozycja_id += 1

pozycje_zamowien_pd = pd.DataFrame(wiersze_pozycji)

# --- konwersja na DataFrame'y Sparka ---------------------------------------
klienci = spark.createDataFrame(klienci_pd)
produkty = spark.createDataFrame(produkty_pd)
pracownicy = spark.createDataFrame(pracownicy_pd)
zamowienia = spark.createDataFrame(zamowienia_pd)
pozycje_zamowien = spark.createDataFrame(pozycje_zamowien_pd)

klienci.show(5)


<a id="sec1"></a>
## 1. MLlib: przegląd DataFrame-based API

Spark ma DWIE biblioteki ML w swojej historii: starą, opartą o RDD (`pyspark.mllib` --
PRZESTARZAŁA, nie używaj jej w nowym kodzie) i nową, opartą o DataFrame
(`pyspark.ml` -- ta, której używamy w tym module). Kluczowe pojęcia w `pyspark.ml`:

- **Transformer** -- obiekt, który przekształca jeden DataFrame w drugi przez metodę
  `.transform()` (np. wytrenowany model, `StringIndexer` po dopasowaniu).
- **Estimator** -- obiekt, który UCZY SIĘ z danych przez metodę `.fit()` i zwraca
  Transformer (np. `LinearRegression` przed treningiem to Estimator, po treningu --
  `LinearRegressionModel`, czyli Transformer).
- **Pipeline** -- łańcuch Transformerów i Estimatorów wykonywanych po kolei jako jeden
  spójny obiekt (sekcja 4).

> ⚡ **To dokładnie te same modele, co w kursie Statystyki**
>
> Matematyka pod spodem `LinearRegression`/`LogisticRegression` w MLlib to DOKŁADNIE to, co poznaliśmy w Modułach 12-13 kursu Statystyki (`statsmodels`/`sklearn`) -- współczynniki, R², dopasowanie metodą najmniejszych kwadratów, funkcja logistyczna. Różnica jest WYŁĄCZNIE w tym, ŻE trening odbywa się rozproszony na wielu maszynach -- interpretacja wyników zostaje identyczna.

<a id="sec2"></a>
## 2. Przygotowanie cech: VectorAssembler

Modele MLlib oczekują JEDNEJ kolumny wektorowej (typu `Vector`) zawierającej wszystkie
cechy liczbowe naraz -- w przeciwieństwie do `sklearn`, gdzie podajesz osobną macierz
`X`. `VectorAssembler` łączy wybrane kolumny w taką jedną kolumnę (domyślnie nazywaną
`features`).

In [ ]:
from pyspark.sql import functions as F

# Przygotowujemy dane na poziomie zamowienia: liczba pozycji, srednia cena, wartosc
zamowienia_cechy = (
    zamowienia.join(klienci, "klient_id")
    .join(pozycje_zamowien, "zamowienie_id")
    .groupBy("zamowienie_id", "klient_id", "segment", "status")
    .agg(
        F.count("*").alias("liczba_pozycji"),
        F.avg("cena_jednostkowa").alias("srednia_cena"),
        F.sum(F.col("ilosc") * F.col("cena_jednostkowa") * (1 - F.col("rabat"))).alias("wartosc"),
    )
)
zamowienia_cechy.show(5)


In [ ]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(
    inputCols=["liczba_pozycji", "srednia_cena"],
    outputCol="cechy_numeryczne",
)
z_cechami = assembler.transform(zamowienia_cechy)
z_cechami.select("zamowienie_id", "liczba_pozycji", "srednia_cena", "cechy_numeryczne").show(5)


> ⚠️ **VectorAssembler nie akceptuje wartości NULL**
>
> Jeśli którakolwiek z kolumn wejściowych ma `NULL`, `VectorAssembler` rzuci błąd przy `.transform()`. Zanim złożysz cechy w wektor, upewnij się, że braki są obsłużone (Moduł 3: `fillna`/`dropna`) -- dokładnie tak samo, jak przy przygotowywaniu danych dla `sklearn`/`statsmodels` w kursie Statystyki.

<a id="sec3"></a>
## 3. Zmienne kategoryczne: StringIndexer i OneHotEncoder

Modele ML potrzebują liczb, nie tekstu -- kolumnę kategoryczną (np. `segment`) trzeba
najpierw zamienić na liczby. Dwa kroki: **`StringIndexer`** (przypisuje każdej unikalnej
wartości tekstowej liczbę całkowitą, od najczęstszej do najrzadszej) i **`OneHotEncoder`**
(zamienia tę liczbę na wektor "gorącojedynkowy", żeby model nie zinterpretował kolejnych
liczb jako fałszywego porządku/odległości).

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

indexer_segmentu = StringIndexer(inputCol="segment", outputCol="segment_index")
z_indeksem = indexer_segmentu.fit(zamowienia_cechy).transform(zamowienia_cechy)
z_indeksem.select("segment", "segment_index").distinct().orderBy("segment_index").show()


In [ ]:
encoder_segmentu = OneHotEncoder(inputCol="segment_index", outputCol="segment_ohe")
z_ohe = encoder_segmentu.fit(z_indeksem).transform(z_indeksem)
z_ohe.select("segment", "segment_index", "segment_ohe").distinct().orderBy("segment_index").show()


> 🧱 **Dlaczego StringIndexer + OneHotEncoder, a nie jeden krok**
>
> W `sklearn` często robi się to jednym wywołaniem (`OneHotEncoder` na tekście bezpośrednio). MLlib rozdziela to na DWA kroki celowo -- `StringIndexer` sam w sobie bywa też użyteczny (np. `LogisticRegression` z natury obsługuje indeksowane etykiety klas bez one-hot encodingu), więc oba kroki są osobnymi, swobodnie łączonymi klockami w `Pipeline` (sekcja 4).

<a id="sec4"></a>
## 4. Pipeline: łączenie kroków w jeden obiekt

Zamiast ręcznie wywoływać `.fit()`/`.transform()` dla każdego kroku z osobna,
`Pipeline` łączy WSZYSTKIE kroki (indexery, encodery, assembler, model) w JEDEN obiekt --
jedno `.fit()` na całym łańcuchu trenuje wszystko po kolei, a jedno `.transform()`
przepuszcza nowe dane przez cały łańcuch.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder

indexer = StringIndexer(inputCol="segment", outputCol="segment_index")
encoder = OneHotEncoder(inputCol="segment_index", outputCol="segment_ohe")
assembler = VectorAssembler(
    inputCols=["liczba_pozycji", "srednia_cena", "segment_ohe"],
    outputCol="features",
)

pipeline_przygotowania = Pipeline(stages=[indexer, encoder, assembler])
model_przygotowania = pipeline_przygotowania.fit(zamowienia_cechy)
dane_gotowe = model_przygotowania.transform(zamowienia_cechy)
dane_gotowe.select("zamowienie_id", "features", "wartosc").show(5, truncate=False)


> ⚡ **Pipeline gwarantuje SPÓJNOŚĆ między treningiem a nowymi danymi**
>
> Największa zaleta `Pipeline` nie jest tylko wygodowa -- gwarantuje, że DOKŁADNIE te same przekształcenia (te same indeksy `StringIndexer`, te same kolumny `VectorAssembler`) zostaną zastosowane do NOWYCH danych w produkcji, jak podczas treningu. Ręczne powtarzanie tych kroków osobno to częste źródło błędów -- *train/serve skew*, gdzie model widzi w produkcji dane przygotowane inaczej niż podczas treningu.

<a id="sec5"></a>
## 5. Regresja liniowa: przewidywanie wartości zamówienia

Zbudujmy pełny `Pipeline` z modelem `LinearRegression` na końcu, przewidujący `wartosc`
zamówienia na podstawie liczby pozycji, średniej ceny i segmentu klienta -- dokładnie
ten sam model co w Module 12 kursu Statystyki, teraz trenowany rozproszenie.

In [ ]:
from pyspark.ml.regression import LinearRegression

trening, test = zamowienia_cechy.randomSplit([0.8, 0.2], seed=42)

model_regresji = LinearRegression(featuresCol="features", labelCol="wartosc")
pipeline_regresji = Pipeline(stages=[indexer, encoder, assembler, model_regresji])

wytrenowany_pipeline = pipeline_regresji.fit(trening)
predykcje = wytrenowany_pipeline.transform(test)
predykcje.select("zamowienie_id", "wartosc", "prediction").show(10)


In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator

ewaluator = RegressionEvaluator(labelCol="wartosc", predictionCol="prediction", metricName="rmse")
rmse = ewaluator.evaluate(predykcje)
r2 = ewaluator.setMetricName("r2").evaluate(predykcje)
print(f"RMSE: {rmse:.2f}")
print(f"R2: {r2:.3f}")


In [ ]:
# Wspolczynniki modelu -- ten sam model liniowy z Modulu 12 kursu Statystyki,
# tylko teraz wewnatrz Pipeline
finalny_model = wytrenowany_pipeline.stages[-1]
print(f"Wspolczynniki: {finalny_model.coefficients}")
print(f"Wyraz wolny (intercept): {finalny_model.intercept:.2f}")


<a id="sec6"></a>
## 6. Regresja logistyczna: przewidywanie anulowania zamówienia

Drugi model z kursu Statystyki (Moduł 13) -- klasyfikacja binarna. Przewidzimy, czy
zamówienie zostanie ANULOWANE, na podstawie tych samych cech. Nasz zestaw danych ma
CELOWO wbudowaną zależność: klienci z segmentu `'Nowy'` anulują znacznie częściej
(Moduł 3 -- tu wreszcie to wykorzystamy w modelu).

In [ ]:
from pyspark.ml.classification import LogisticRegression

zamowienia_z_etykieta = zamowienia_cechy.withColumn(
    "czy_anulowane", (F.col("status") == "Anulowane").cast("double")
)

trening_log, test_log = zamowienia_z_etykieta.randomSplit([0.8, 0.2], seed=42)

model_logistyczny = LogisticRegression(featuresCol="features", labelCol="czy_anulowane")
pipeline_logistyczny = Pipeline(stages=[indexer, encoder, assembler, model_logistyczny])

wytrenowany_logistyczny = pipeline_logistyczny.fit(trening_log)
predykcje_log = wytrenowany_logistyczny.transform(test_log)
predykcje_log.select("zamowienie_id", "czy_anulowane", "probability", "prediction").show(10, truncate=False)


In [ ]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

ewaluator_log = BinaryClassificationEvaluator(
    labelCol="czy_anulowane", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
auc = ewaluator_log.evaluate(predykcje_log)
print(f"AUC (pole pod krzywa ROC): {auc:.3f}")


> ⚠️ **AUC z Modulu 9/13 kursu Statystyki -- ta sama miara, ten sam sposob czytania**
>
> `areaUnderROC` to DOKŁADNIE ta sama miara AUC, którą poznaliśmy przy regresji logistycznej w kursie Statystyki -- 0.5 oznacza model nie lepszy niż losowe zgadywanie, 1.0 oznacza perfekcyjne rozróżnianie klas. Interpretacja jest identyczna niezależnie od tego, czy model wytrenowano w `sklearn` czy w MLlib.

<a id="sec7"></a>
## 7. Podsumowanie i ćwiczenia

W tym module poznaliśmy:
- `pyspark.ml` (DataFrame-based) kontra przestarzałe `pyspark.mllib` (RDD-based),
- pojęcia Transformer/Estimator/Pipeline,
- `VectorAssembler` -- łączenie kolumn liczbowych w jedną kolumnę `features`,
- `StringIndexer` + `OneHotEncoder` -- kodowanie zmiennych kategorycznych,
- `Pipeline` -- łączenie wszystkich kroków w jeden spójny, powtarzalny obiekt,
- `LinearRegression` i `LogisticRegression` w MLlib -- te same modele co w kursie Statystyki, trenowane rozproszenie.

### 📝 Ćwiczenie 1: VectorAssembler na produktach

Użyj `VectorAssembler`, żeby złączyć kolumnę `cena` z `produkty` w kolumnę `features` (jednoelementowy wektor).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=["cena"], outputCol="features")
assembler.transform(produkty).select("produkt_id", "cena", "features").show(5)
```

</details>

### 📝 Ćwiczenie 2: StringIndexer na kategorii produktu

Użyj `StringIndexer` na kolumnie `kategoria` z `produkty` i wypisz przypisanie kategorii do indeksów (posortowane po indeksie).

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml.feature import StringIndexer

indexer = StringIndexer(inputCol="kategoria", outputCol="kategoria_index")
indexer.fit(produkty).transform(produkty).select(
    "kategoria", "kategoria_index"
).distinct().orderBy("kategoria_index").show()
```

</details>

### 📝 Ćwiczenie 3: Prosty Pipeline regresji

Zbuduj `Pipeline` z `VectorAssembler` (cecha: `liczba_pozycji`) i `LinearRegression` (`labelCol="wartosc"`) na `zamowienia_cechy`, wytrenuj na całym zbiorze (bez podziału trening/test) i wypisz `coefficients`/`intercept`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression

assembler = VectorAssembler(inputCols=["liczba_pozycji"], outputCol="features")
model = LinearRegression(featuresCol="features", labelCol="wartosc")
pipeline = Pipeline(stages=[assembler, model])

wytrenowany = pipeline.fit(zamowienia_cechy)
finalny = wytrenowany.stages[-1]
print(f"Wspolczynniki: {finalny.coefficients}")
print(f"Intercept: {finalny.intercept:.2f}")
```

</details>

### 📝 Ćwiczenie 4: Ewaluacja RMSE bez podziału na train/test

Wytrenuj `Pipeline` z ćwiczenia 3, przepuść przez niego CAŁY zbiór `zamowienia_cechy` i policz RMSE przez `RegressionEvaluator`.

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml.evaluation import RegressionEvaluator

predykcje = wytrenowany.transform(zamowienia_cechy)
ewaluator = RegressionEvaluator(labelCol="wartosc", predictionCol="prediction", metricName="rmse")
print(f"RMSE: {ewaluator.evaluate(predykcje):.2f}")
```

</details>

> 🔥 **Wyzwanie: pełny Pipeline z wieloma cechami i porównaniem AUC**
>
> Zbuduj DWA modele `LogisticRegression` przewidujące anulowanie zamówienia: (1) tylko z cechą `liczba_pozycji`, (2) z pełnym zestawem cech z sekcji 6 (`liczba_pozycji`, `srednia_cena`, `segment` przez one-hot encoding). Porównaj AUC obu modeli na tym samym zbiorze testowym -- czy dodanie segmentu poprawia model?

<details>
<summary>Pokaż rozwiązanie</summary>

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator

dane = zamowienia_cechy.withColumn(
    "czy_anulowane", (F.col("status") == "Anulowane").cast("double")
)
trening, test = dane.randomSplit([0.8, 0.2], seed=42)

# Model 1: tylko liczba_pozycji
assembler_prosty = VectorAssembler(inputCols=["liczba_pozycji"], outputCol="features")
model_prosty = LogisticRegression(featuresCol="features", labelCol="czy_anulowane")
pipeline_prosty = Pipeline(stages=[assembler_prosty, model_prosty])
predykcje_prosty = pipeline_prosty.fit(trening).transform(test)

# Model 2: pelny zestaw cech
indexer = StringIndexer(inputCol="segment", outputCol="segment_index")
encoder = OneHotEncoder(inputCol="segment_index", outputCol="segment_ohe")
assembler_pelny = VectorAssembler(
    inputCols=["liczba_pozycji", "srednia_cena", "segment_ohe"], outputCol="features"
)
model_pelny = LogisticRegression(featuresCol="features", labelCol="czy_anulowane")
pipeline_pelny = Pipeline(stages=[indexer, encoder, assembler_pelny, model_pelny])
predykcje_pelny = pipeline_pelny.fit(trening).transform(test)

ewaluator = BinaryClassificationEvaluator(
    labelCol="czy_anulowane", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
)
print(f"AUC (tylko liczba_pozycji): {ewaluator.evaluate(predykcje_prosty):.3f}")
print(f"AUC (pelny zestaw cech): {ewaluator.evaluate(predykcje_pelny):.3f}")
```

Poniewaz nasz zestaw danych ma celowo wbudowana zaleznosc segment-anulowanie (Modul 3), model z segmentem powinien osiagac wyzsze AUC -- dokladnie to, czego oczekujemy od dobrej cechy predykcyjnej.

</details>

## Co dalej?

W **Module 13** zajmiemy się dokładniejszą oceną modeli: podział train/test/validation,
`CrossValidator` i `ParamGridBuilder` do strojenia hiperparametrów -- w pełni
zautomatyzowane przeszukiwanie najlepszej konfiguracji modelu.